# Sensor characterization (Bluesky run)

Calibrates both thermistors and both heater resistors against a HOBO MX1101 in the chamber, from the 25-26 Oct 2026 passive resistance sweep. Readings are used only where the reference and the device have both settled.

This notebook reads the sweep from its Bluesky run in `runs/`, which `convert_legacy_data.py` made from the files in `data/`. New sweeps from `acquisition.py` land in the same folder with the same data keys. The analysis is the same as `sensor_characterization.ipynb`, and the last cell checks that both give the same fits. It writes no files, so `results/` stays as that notebook left it.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cryojane_bench.logs import read_channel
from cryojane_bench.plateaus import find_plateaus
from cryojane_bench.plotting import BLUE, MUTED, ORANGE, apply_style
from cryojane_bench.runs import load_run
from cryojane_bench.thermistor import fit_beta, fit_steinhart_hart
from matplotlib.ticker import ScalarFormatter

apply_style()
RUNS, RESULTS = Path("runs"), Path("results")
COLORS = {"0": BLUE, "1": ORANGE}
RUN_UID = "ec0ee2da"  # The converted 25-26 Oct sweep.

run = load_run(next(RUNS.glob(f"*_{RUN_UID}.jsonl")))
readings = run.table("primary")
units = run.data_keys("primary")["gdm_reading"]["units"]
results = pd.DataFrame({
    "timestamp": readings.index,
    "channel": readings.channel_name.to_numpy(),
    "value": readings.gdm_reading.to_numpy(),
})
reference_c = run.table("chamber_reference").hobo_temperature.rename("temperature_c")
sensors = {machine: read_channel(results, f"Sensor ({machine})") for machine in COLORS}
heaters = {machine: read_channel(results, f"Heater ({machine})") for machine in COLORS}

print(f"Run {run.start['uid']}, plan {run.start['plan_name']}, exit {run.stop['exit_status']}")
print(f"Sources: {', '.join(source['path'] for source in run.start['converted_from'])}")
print(f"Meter configuration: {run.configuration('primary')['gdm']}, readings in {units}")
print(f"Reference: {run.configuration('chamber_reference')['hobo']['hobo_model']}, "
      f"{reference_c.min():.2f} to {reference_c.max():.2f} °C, {len(reference_c)} samples")
print(f"Sweep {results.timestamp.min()} to {results.timestamp.max()}, {len(results)} readings")


## Plateaus

The settling check on the device needs only a rough temperature scale, so it uses a nominal curve built from each channel's median.

In [ ]:
NOMINAL_BETA_K = 3700.0
NOMINAL_HEATER_TCR = 3.9e-3  # Deliberately loose. The measured TCR is about 30x lower, and a tight check rejects most heater steps.

def sensor_plateaus(readings):
    median = readings.median()
    return find_plateaus(readings, reference_c, lambda ohms: 1.0 / (np.log(ohms / median) / NOMINAL_BETA_K + 1.0 / 263.15) - 273.15)

def heater_plateaus(readings):
    median = readings.median()
    return find_plateaus(readings, reference_c, lambda ohms: (ohms / median - 1.0) / NOMINAL_HEATER_TCR)

sensor_steps = {machine: sensor_plateaus(readings) for machine, readings in sensors.items()}
heater_steps = {machine: heater_plateaus(readings) for machine, readings in heaters.items()}
for machine in COLORS:
    print(f"Machine {machine}: {len(sensor_steps[machine])} sensor plateaus, {len(heater_steps[machine])} heater plateaus")

all_steps = {f"Sensor ({m})": s for m, s in sensor_steps.items()} | {f"Heater ({m})": s for m, s in heater_steps.items()}
plateaus = pd.concat(all_steps, names=["channel", "plateau"])

fig, (ax_reference, ax_sensor, ax_heater) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax_reference.plot(reference_c.index, reference_c, color=MUTED)
ax_reference.set_ylabel("Chamber (°C)")
ax_reference.set_title("Passive sweep. Shaded stretches are the plateaus used for the fits.")
for machine, color in COLORS.items():
    for ax, readings, steps, scale in ((ax_sensor, sensors[machine], sensor_steps[machine], 1e-3), (ax_heater, heaters[machine], heater_steps[machine], 1.0)):
        trace = readings.resample("2min").median()
        ax.plot(trace.index, trace * scale, color=color, label=f"Machine {machine}")
        for _, step in steps.iterrows():
            ax.axvspan(step.start, step.end, color=color, alpha=0.08, linewidth=0)
ax_sensor.set_ylabel("Thermistor (kΩ)")
ax_heater.set_ylabel("Heater (Ω)")
ax_sensor.legend()
ax_heater.legend()
ax_heater.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## Thermistor fits

Steinhart-Hart is the curve to use. The beta fit is here to show how much the third term buys.

In [ ]:
rows = []
sh_fits = {}
for machine, steps in sensor_steps.items():
    sh = fit_steinhart_hart(steps.value, steps.reference_c)
    beta = fit_beta(steps.value, steps.reference_c)
    sh_fits[machine] = sh
    sh_error = sh.temperature_c(steps.value) - steps.reference_c
    beta_error = beta.temperature_c(steps.value) - steps.reference_c
    rows.append({
        "channel": f"Sensor ({machine})", "plateaus": len(steps),
        "reference_min_c": steps.reference_c.min(), "reference_max_c": steps.reference_c.max(),
        "sh_a": sh.a, "sh_b": sh.b, "sh_c": sh.c,
        "sh_rms_c": np.sqrt(np.mean(sh_error**2)), "sh_worst_c": sh_error.abs().max(),
        "r25_ohm": beta.r25_ohm, "beta_k": beta.beta_k, "beta_rms_c": np.sqrt(np.mean(beta_error**2)),
    })
thermistor_summary = pd.DataFrame(rows).set_index("channel")
print(thermistor_summary.T.to_string(float_format=lambda x: f"{x:.6g}"))

fig, (ax_curve, ax_error) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
for machine, steps in sensor_steps.items():
    color, fit = COLORS[machine], sh_fits[machine]
    temperatures = np.linspace(steps.reference_c.min(), steps.reference_c.max(), 200)
    ax_curve.plot(temperatures, fit.resistance_ohm(temperatures) / 1e3, color=color, label=f"Machine {machine} fit")
    ax_curve.plot(steps.reference_c, steps.value / 1e3, "o", color=color, markersize=7, markeredgecolor="#fcfcfb")
    ax_error.plot(steps.reference_c, fit.temperature_c(steps.value) - steps.reference_c, "o-", color=color, markersize=7, markeredgecolor="#fcfcfb", label=f"Machine {machine}")
ax_error.axhline(0, color=MUTED, linewidth=1)
ax_curve.set_yscale("log")
ax_curve.yaxis.set_major_formatter(ScalarFormatter())
ax_curve.yaxis.set_minor_formatter(ScalarFormatter())
ax_curve.set_ylabel("Thermistor resistance (kΩ, log scale)")
ax_curve.set_title("Steinhart-Hart fits against the chamber reference")
ax_curve.legend()
ax_error.set(xlabel="Chamber temperature (°C)", ylabel="Fit error (°C)")
ax_error.legend()
fig.tight_layout()
plt.show()

## Sensor 1 cold-end noise

Sensor 1's four coldest plateaus have several times the scatter of the rest, with spikes of 2-3 kΩ, and they account for most of its larger fit error. Plateaus use the median, which limits the effect but does not remove it. The cause is not known. A marginal connection is one candidate, because heater 1 misbehaves over part of the same stretch.

In [ ]:
scatter = pd.DataFrame({
    "chamber_c": sensor_steps["1"].reference_c,
    **{f"machine_{machine}_ohm": steps.value_std for machine, steps in sensor_steps.items()},
})
print("Reading scatter on each plateau (standard deviation), coldest first")
print(scatter.round(1).to_string(index=False))

## Heater resistance

Linear fit, R = R0 (1 + α T), with T in °C. Heater 1 twice moves away from its fit, by up to about 0.4 Ω around 02:30 and 0.25 Ω around 07:30, then recovers. Temperature does not explain this, and heater 0 shows nothing similar. The plateau check drops those stretches, so heater 1 has fewer plateaus and a much larger residual.

In [ ]:
rows = []
for machine, steps in heater_steps.items():
    slope, intercept = np.polyfit(steps.reference_c, steps.value, 1)
    residual = steps.value - (intercept + slope * steps.reference_c)
    rows.append({
        "channel": f"Heater ({machine})", "plateaus": len(steps),
        "reference_min_c": steps.reference_c.min(), "reference_max_c": steps.reference_c.max(),
        "r_at_0c_ohm": intercept, "tcr_ppm_per_c": 1e6 * slope / intercept, "linear_rms_mohm": 1e3 * np.sqrt(np.mean(residual**2)),
    })
heater_summary = pd.DataFrame(rows).set_index("channel")
print(heater_summary.T.to_string(float_format=lambda x: f"{x:.5g}"))

fig, ax = plt.subplots(figsize=(10, 4))
for machine, readings in heaters.items():
    trace = readings.resample("2min").median()
    fit = heater_summary.loc[f"Heater ({machine})"]
    aligned = reference_c.reindex(trace.index, method="nearest")
    expected = fit.r_at_0c_ohm * (1 + fit.tcr_ppm_per_c * 1e-6 * aligned)
    ax.plot(trace.index, (trace - expected) * 1e3, color=COLORS[machine], linewidth=1.5, label=f"Machine {machine}")  # The start is the chamber cooling down.
ax.set_ylim(-150, 450)
ax.set_ylabel("Measured − linear fit (mΩ)")
ax.set_title("Heater 1 twice moves away from its fit and recovers. Heater 0 does not.")
ax.legend()
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

fit_summary = pd.concat([thermistor_summary, heater_summary])

## Same answer as the text-log notebook

Compares this notebook's fits and plateaus with the files `sensor_characterization.ipynb` wrote.

In [ ]:
saved_fits = pd.read_csv(RESULTS / "fit_summary.csv", index_col="channel")
saved_plateaus = pd.read_csv(RESULTS / "plateaus.csv", index_col=["channel", "plateau"])
fit_difference = (fit_summary[saved_fits.columns] - saved_fits).abs().max().max()
plateau_difference = (plateaus[["value", "reference_c"]] - saved_plateaus[["value", "reference_c"]]).abs().max().max()
print(f"Largest fit summary difference: {fit_difference:.3g}")
print(f"Largest plateau value difference: {plateau_difference:.3g}")
assert np.allclose(fit_summary[saved_fits.columns], saved_fits, rtol=1e-9, atol=1e-12, equal_nan=True)
assert plateaus.index.equals(saved_plateaus.index)
